# Data Loading: MNIST Binary Classification (3 vs 6)

Downloads MNIST on first use (cached in `data/raw/mnist.npz`), filters to
digits 3 and 6, subsamples with a data seed, and reduces the images to the
fixed low-dimensional PCA representation consumed by the QNN encoder.

In [2]:
import os
import sys
from pathlib import Path

# Run from the project root regardless of the notebook's directory.
ROOT = Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")


'1'

## Load Binary MNIST

In [3]:
from src.data import load_mnist_binary

X_train, y_train, X_test, y_test = load_mnist_binary(
    digit1=3, digit2=6, train_size=1000, test_size=200, data_seed=42
)
print(f"Train: {X_train.shape}, labels {y_train.shape}")
print(f"Test:  {X_test.shape}, labels {y_test.shape}")
print(f"Label distribution (train): 0={int((y_train == 0).sum())}, "
      f"1={int((y_train == 1).sum())}")


Train: (1000, 784), labels (1000,)
Test:  (200, 784), labels (200,)
Label distribution (train): 0=546, 1=454


## Downsample + PCA

28x28 -> 4x4 bilinear -> 16 features -> PCA to
`n_components = n_qubits` (fitted on the training split only).

In [4]:
from src.data import prepare_features

N_QUBITS = 8
X_train_r, X_test_r, pca_info = prepare_features(
    X_train, X_test, n_components=N_QUBITS, image_size=(4, 4)
)
print(f"Reduced train: {X_train_r.shape}, test: {X_test_r.shape}")
print(f"Explained variance: {pca_info['cumulative_explained_variance']:.4f}")


Reduced train: (1000, 8), test: (200, 8)
Explained variance: 1.0000


## Encode to Rotation Angles

Scales each feature to `[0, pi]`.

In [5]:
from src.data import encode_data_for_qnn

angles_train = encode_data_for_qnn(X_train_r)
print(f"Encoded train: {angles_train.shape}")
print(f"Angle range: [{angles_train.min():.4f}, {angles_train.max():.4f}]")


Encoded train: (1000, 8)
Angle range: [0.0000, 3.1416]


## Sample Images

In [6]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for row, (label, idx) in enumerate([(0, 0), (1, 1)]):
    xs = X_train[y_train == label][:5]
    for col, img in enumerate(xs):
        axes[row, col].imshow(img.reshape(28, 28), cmap="gray")
        axes[row, col].axis("off")
        axes[row, col].set_title(f"digit {3 if label == 0 else 6}")
plt.tight_layout()
plt.show()


<Figure size 1200x500 with 10 Axes>

## Dataset Summary

In [7]:
summary = {
    "train_size": len(X_train),
    "test_size": len(X_test),
    "pixels_per_image": X_train.shape[1],
    "n_pca_components": pca_info["n_components"],
    "cumulative_explained_variance": round(pca_info["cumulative_explained_variance"], 4),
}
for k, v in summary.items():
    print(f"{k}: {v}")


train_size: 1000
test_size: 200
pixels_per_image: 784
n_pca_components: 8
cumulative_explained_variance: 1.0
